# VigiLens on Google Colab

Normal-only training of the RGB+IR anomaly detector, then calibration and evaluation.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU` (any GPU works).

Pick a dataset in cell 3:

| `DATASET` | What it is | Modalities | Anomaly labels | Use it for |
|---|---|---|---|---|
| `"vidllvip"` | Paired visible/IR pedestrian videos (`vi/`, `ir/` with matching names) | RGB + IR | none | Training the multimodal model; qualitative scores |
| `"kaist"` | KAIST multispectral pedestrian sequences (`setXX/VYYY/visible`, `lwir`) | RGB + IR | none | Larger normal training corpus |
| `"ped2"` | UCSD Ped2 walkway (bikes/carts/skaters are anomalies) | visible only | frame-level | **Quantitative** AUC / AP / FPR |

None of the RGB+IR sets ship anomaly labels, so a real number for the multimodal model needs
your own labeled clips (cell 11 shows the folder layout). `ped2` gives a standard benchmark
number with the model running in RGB-only mode.

Upload the dataset archive (`.zip` / `.tar` / `.tar.gz`) to Google Drive first; the notebook
copies it to the Colab disk, because reading thousands of frames straight from Drive is slow.

In [ ]:
# 1. GPU check
import os, sys, json, shutil, subprocess
from pathlib import Path
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: precompute will be very slow. Switch the runtime to a GPU.")


def run(cmd):
    # Run a shell command, stream its output, and stop the notebook on failure.
    print("$", cmd)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {cmd}")

In [ ]:
# 2. Get the code.  Point REPO_URL/BRANCH at the fork/branch that has the fixes.
REPO_URL = "https://github.com/Gauranga025/VigiLens.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/VigiLens")

if not PROJECT_DIR.exists():
    run(f"git clone --branch {BRANCH} {REPO_URL} {PROJECT_DIR}")
os.chdir(PROJECT_DIR)
run("git log -1 --oneline")
# Colab already has CUDA torch/torchvision; pip keeps them.
run("pip install -q -r requirements.txt")

In [ ]:
# 3. Settings
DATASET = "vidllvip"        # "vidllvip" | "kaist" | "ped2"
# Archive on Google Drive (after mounting) or any local path / URL.
DATA_ARCHIVE = "/content/drive/MyDrive/datasets/VidLLVIP.zip"
EPOCHS = 30
BATCH_SIZE = 32
SEQ_LEN = 16
SMOKE_TEST = True           # run a 2-epoch mini pipeline first
SAVE_TO_DRIVE = True
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/VigiLens_runs") / DATASET

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Copy + extract the dataset onto the Colab disk
RAW = Path("/content/raw") / DATASET
if not RAW.exists() or not any(RAW.iterdir()):
    RAW.mkdir(parents=True, exist_ok=True)
    archive = DATA_ARCHIVE
    if archive.startswith("http"):
        run(f"wget -q -O /content/archive.bin '{archive}'")
        archive = "/content/archive.bin"
    elif not Path(archive).exists():
        raise FileNotFoundError(f"Upload the dataset archive to Drive and set DATA_ARCHIVE (not found: {archive})")
    if archive.endswith(".zip"):
        run(f"unzip -q '{archive}' -d {RAW}")
    else:
        run(f"tar -xf '{archive}' -C {RAW}")
run(f"find {RAW} -maxdepth 3 -type d | head -20")

In [ ]:
# 5. Convert to Data/<split>/{rgb,ir}/*.mp4  (+ Evaluation/ for ped2)
DATA_DIR = Path(f"Data_{DATASET}")
EVAL_DIR = Path(f"Evaluation/{DATASET}")

def find_dir_with(root, *names):
    # Return the first folder under root that contains all the given sub-folders.
    for path in [root, *sorted(p for p in root.rglob("*") if p.is_dir())]:
        if all((path / name).exists() for name in names):
            return path
    raise FileNotFoundError(f"No folder with {names} under {root}")

if DATASET == "vidllvip":
    src = find_dir_with(RAW, "vi", "ir")
    # Clips of one source video must stay in one split.  If the script reports
    # too few sources, set --source-regex for your file naming.
    run(f"python scripts/prepare_vidllvip.py --input '{src}' --output {DATA_DIR} "
        f"--val-ratio 0.15 --test-ratio 0.15 --seed 42 --clean")
elif DATASET == "kaist":
    # Official KAIST split: sets 00-05 train, 06-11 test; hold out set 05 for validation.
    run(f"python scripts/prepare_kaist.py --input '{RAW}' --output {DATA_DIR} "
        f"--sets 00,01,02,03,04,05 --val-sets 05 --clean")
elif DATASET == "ped2":
    run(f"python scripts/prepare_ucsd_ped2.py --input '{RAW}' --output {DATA_DIR} "
        f"--eval-root {EVAL_DIR} --val-videos 2 --clean")
else:
    raise ValueError(DATASET)

for split in ("train", "val", "test"):
    print(split, len(list((DATA_DIR / split / "rgb").glob("*.mp4"))), "RGB videos")

### Optional smoke test
Runs the whole pipeline on 60 frames per clip for 2 epochs so mistakes show up in a minute, not an hour.

In [ ]:
# 6. Smoke test (fast)
if SMOKE_TEST:
    run(f"rm -rf cache_smoke checkpoints_smoke")
    run(f"python -m training.precompute --data-dir {DATA_DIR} --output cache_smoke "
        f"--device {DEVICE} --max-frames 60 --splits train,val")
    run(f"python -m training.train --cache-dir cache_smoke --epochs 2 --batch-size 8 "
        f"--seq-len {SEQ_LEN} --device {DEVICE} --checkpoint-dir checkpoints_smoke --calibrate")
    print("Smoke test OK")

In [ ]:
# 7. Precompute features once (YOLO person masks + frozen ResNet50).  This is
#    the slow step; caches are rebuilt automatically if settings change.
run(f"python -m training.precompute --data-dir {DATA_DIR} --output cache/{DATASET} "
    f"--device {DEVICE} --batch-size 32 --splits train,val")

In [ ]:
# 8. Train fusion + LSTM on the cached features, then calibrate the BEST
#    checkpoint on the held-out normal validation videos.
CKPT_DIR = Path(f"checkpoints/{DATASET}")
run(f"python -m training.train --cache-dir cache/{DATASET} --epochs {EPOCHS} "
    f"--batch-size {BATCH_SIZE} --seq-len {SEQ_LEN} --device {DEVICE} "
    f"--checkpoint-dir {CKPT_DIR} --calibrate")

In [ ]:
# 9. Training curves.  fused_std should NOT fall towards zero (that would be
#    representation collapse); val_loss should go down and level off.
import pandas as pd
history = pd.DataFrame(json.loads((CKPT_DIR / "training_history.json").read_text()))
display(history.tail())
history.plot(x="epoch", y=["train_loss", "val_loss"], logy=True, title="loss")
history.plot(x="epoch", y="fused_std", title="fused feature spread")
cal = json.loads((CKPT_DIR / "calibration.json").read_text())
print("raw threshold:", cal["threshold_raw"])
print("threshold for moving average window 10:", cal["thresholds"]["moving_average"]["10"])

In [ ]:
# 10. Evaluation
SMOOTHING = 10   # must match the app's "Window Size" to compare like with like

if DATASET == "ped2":
    # Labeled benchmark: frame-level AUC / AP / FPR.
    run(f"python -m training.precompute --manifest {EVAL_DIR}/manifest.json "
        f"--output cache/{DATASET}/eval --device {DEVICE}")
    run(f"python -m training.evaluate --manifest {EVAL_DIR}/manifest.json "
        f"--cache-dir cache/{DATASET}/eval --checkpoint {CKPT_DIR}/best_model.pth "
        f"--calibration {CKPT_DIR}/calibration.json --output-dir {EVAL_DIR}/results "
        f"--device {DEVICE} --smoothing-window {SMOOTHING}")
    results = json.loads((EVAL_DIR / "results" / "results.json").read_text())
    print(json.dumps({k: results["metrics"][k] for k in ("auc", "ap", "precision", "recall", "f1", "fpr")}, indent=2))
    print("(metrics_at_best_f1_threshold is tuned on the test set itself: an optimistic upper bound)")
else:
    # No labels: score held-out test clips and look at the score curves.
    import matplotlib.pyplot as plt
    from models.anomaly_model import VigiLensModel
    from training.calibration import threshold_for
    from training.scoring import score_cache_file

    if (DATA_DIR / "test" / "rgb").exists():
        run(f"python -m training.precompute --data-dir {DATA_DIR} --output cache/{DATASET} "
            f"--device {DEVICE} --splits test")
    model = VigiLensModel(device=DEVICE, load_extractors=False)
    model.load_checkpoint(str(CKPT_DIR / "best_model.pth"))
    model.eval()
    threshold, source = threshold_for(cal, "moving_average", SMOOTHING)
    test_caches = sorted(Path(f"cache/{DATASET}/test").glob("*.pt"))
    print(f"{len(test_caches)} test clips, threshold {threshold:.4f} ({source})")
    flagged = 0
    for path in test_caches[:6]:
        result = score_cache_file(model, path, smoothing_window=SMOOTHING, device=DEVICE)
        scores = result["smoothed_scores"][1:]
        flagged += int((scores >= threshold).sum())
        plt.figure(figsize=(10, 2)); plt.plot(scores); plt.axhline(threshold, color="r")
        plt.title(result["sequence_id"]); plt.show()
    print("These clips are normal footage, so most frames should stay below the red line.")

### Optional: your own labeled RGB+IR clips (real numbers for the multimodal model)

Put clips in Drive like this, then run the next cell:

```
Evaluation/custom/
  normal/rgb/*.mp4   normal/ir/*.mp4        (same file stems in rgb/ and ir/, e.g. walk01_rgb.mp4 / walk01_ir.mp4)
  anomaly/rgb/*.mp4  anomaly/ir/*.mp4
  segments.json      {"fight01": [[120, 260]], ...}   anomalous frame ranges [start, end)
```
Anomaly clips without an entry in `segments.json` are treated as anomalous in every frame.

In [ ]:
# 11. Evaluate on custom labeled clips (skips if the folder does not exist)
CUSTOM = Path("/content/drive/MyDrive/VigiLens_eval/custom")
if CUSTOM.exists():
    segments = CUSTOM / "segments.json"
    seg_arg = f"--segments '{segments}'" if segments.exists() else "--anomaly-all"
    run(f"python scripts/make_eval_manifest.py --root '{CUSTOM}' --output Evaluation/custom/manifest.json {seg_arg}")
    run(f"python -m training.precompute --manifest Evaluation/custom/manifest.json "
        f"--output cache/custom_eval --device {DEVICE}")
    run(f"python -m training.evaluate --manifest Evaluation/custom/manifest.json "
        f"--cache-dir cache/custom_eval --checkpoint {CKPT_DIR}/best_model.pth "
        f"--calibration {CKPT_DIR}/calibration.json --output-dir Evaluation/custom/results "
        f"--device {DEVICE} --smoothing-window {SMOOTHING}")
else:
    print("No custom evaluation set at", CUSTOM)

In [ ]:
# 12. Save checkpoint + calibration (+ results) to Drive.  Download
#     best_model.pth and calibration.json into the repo's checkpoints/ folder
#     to use them in the Streamlit app.
if SAVE_TO_DRIVE:
    DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("best_model.pth", "last_model.pth", "calibration.json", "training_history.json"):
        if (CKPT_DIR / name).exists():
            shutil.copy2(CKPT_DIR / name, DRIVE_RUN_DIR / name)
    if (EVAL_DIR / "results").exists():
        shutil.copytree(EVAL_DIR / "results", DRIVE_RUN_DIR / "results", dirs_exist_ok=True)
    print("Saved to", DRIVE_RUN_DIR)

## Using the trained model in the app

Copy `best_model.pth` and `calibration.json` from `VigiLens_runs/<dataset>/` on Drive into
`checkpoints/` and run `streamlit run app.py`.  The app checks that the calibration was computed
for that exact checkpoint, and picks the threshold that matches the smoothing window you select.